# Notebook 2: Data Cleaning & Feature Engineering

## Objective

This notebook prepares the Olist E-Commerce dataset for SQL analysis and dashboard development.

The workflow includes:

- Assessing data quality
- Handling missing values
- Removing duplicates
- Correcting data types
- Engineering new analytical features
- Exporting cleaned datasets

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

products = pd.read_csv("../data/raw/olist_products_dataset.csv")

sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

In [4]:
def data_quality_report(df, name):

    report = {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Memory (MB)": round(df.memory_usage(deep=True).sum()/1024**2,2)
    }

    return report

In [5]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

quality = pd.DataFrame(
    [data_quality_report(df, name)
     for name, df in datasets.items()]
)

quality

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Memory (MB)
0,Customers,99441,5,0,0,29.62
1,Orders,99441,8,4908,0,58.97
2,Order Items,112650,7,0,0,39.43
3,Payments,103886,5,0,0,17.81
4,Reviews,99224,7,145903,0,42.75
5,Products,32951,9,2448,0,6.79
6,Sellers,3095,4,0,0,0.66
7,Geolocation,1000163,5,0,261831,146.09
8,Category Translation,71,2,0,0,0.01


# Cleaning Decision Log

This section documents the rationale behind every cleaning decision.

Each missing value or duplicate will be investigated before applying any transformation.

The goal is to preserve valid business information while preparing the dataset for analysis.

In [6]:
orders.isnull().sum().sort_values(ascending=False)

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_estimated_delivery_date       0
dtype: int64

In [7]:
orders.isnull().mean().mul(100).round(2)

order_id                         0.00
customer_id                      0.00
order_status                     0.00
order_purchase_timestamp         0.00
order_approved_at                0.16
order_delivered_carrier_date     1.79
order_delivered_customer_date    2.98
order_estimated_delivery_date    0.00
dtype: float64

In [8]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [9]:
orders[
    orders["order_delivered_customer_date"].isnull()
]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [10]:
orders[
    orders["order_approved_at"].isnull()
]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [11]:
orders[
    orders["order_delivered_carrier_date"].isnull()
]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

## Findings – Orders Dataset

### Missing Values Investigation

The Orders dataset contains missing values in three timestamp columns:

- order_approved_at
- order_delivered_carrier_date
- order_delivered_customer_date

After analyzing `order_status`, these missing values are largely explained by the order lifecycle.

Observations:

- Cancelled, unavailable, processing, created, approved, and invoiced orders naturally lack delivery-related timestamps because they never reached those stages.
- A very small number of delivered orders have missing timestamps (8 missing customer delivery dates, 14 missing approval dates, and 2 missing carrier dates). These appear to be data inconsistencies rather than expected business behavior.

Decision:

The missing values will be retained during the initial cleaning phase because they mostly represent valid business scenarios rather than poor data quality.

In [12]:
orders.select_dtypes(include="object").columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

In [13]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

In [14]:
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [15]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [16]:
customers.nunique()

customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64

In [17]:
customers["customer_unique_id"].duplicated().sum()

np.int64(3345)

In [18]:
customers[
    customers["customer_unique_id"].duplicated(keep=False)
].sort_values("customer_unique_id").head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
35608,24b0e2bd287e47d54d193e7bbb51103f,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
19299,1afe8a9c67eec3516c09a8bdcc539090,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
20023,1b4a75b3478138e99902678254b260f4,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
22066,f6efe5d5c7b85e12355f9d5c3db46da2,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
72451,49cf243e0d353cd418ca77868e24a670,004b45ec5c64187465168251cd1c9c2f,57055,maceio,AL
87012,d95f60d70d9ea9a7fe37c53c931940bb,004b45ec5c64187465168251cd1c9c2f,57035,maceio,AL
36269,8ac44e9c15d396b8c3c7cbab0fff4536,0058f300f57d7b93c477a131a59b36c3,41370,salvador,BA
61403,f530197ea86ced9488a03d055e118ebf,0058f300f57d7b93c477a131a59b36c3,40731,salvador,BA
87414,cbb68c721ba9ddb30d8a490cc1897fa1,00a39521eb40f7012db50455bf083460,72595,brasilia,DF
54881,876356df457f952458a764348e1858bc,00a39521eb40f7012db50455bf083460,72595,brasilia,DF


## Findings – Customers Dataset

### Customer Identifiers

The Customers table contains two identifiers:

- `customer_id` identifies a customer record associated with a specific order.
- `customer_unique_id` identifies the actual customer across multiple purchases.

Observations:

- Every `customer_id` is unique.
- `customer_unique_id` appears multiple times, indicating repeat customers.
- Different customer records for the same individual may have different ZIP codes, suggesting address updates or different delivery locations over time.

Business Insight:

Using two identifiers preserves the historical shipping information for each order while still allowing repeat customers to be analyzed correctly.

# 6. Product Dataset Exploration

In [19]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [20]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [21]:
products.nunique()

product_id                    32951
product_category_name            73
product_name_lenght              66
product_description_lenght     2960
product_photos_qty               19
product_weight_g               2204
product_length_cm                99
product_height_cm               102
product_width_cm                 95
dtype: int64

In [22]:
products.isnull().sum().sort_values(ascending=False)

product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
product_id                      0
dtype: int64

In [23]:
products[
    products["product_category_name"].isnull()
].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [24]:
products[
    products["product_category_name"].isnull()
].isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [25]:
products[
    products["product_weight_g"].isnull()
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Findings

- 610 products are missing category name, description length, name length, and photo count.
- Most of these products still contain weight and dimension information.
- This suggests the products existed physically but their catalog metadata was incomplete.
- Since these products are referenced in transactions, they will be retained during analysis.
- Only one additional product has missing physical dimensions outside this incomplete group.

In [26]:
print("Shape:", sellers.shape)

print("\nColumns:")
print(sellers.columns.tolist())

print("\nData Types:")
display(sellers.dtypes.to_frame("Data Type"))

print("\nMissing Values:")
display(sellers.isnull().sum().to_frame("Missing"))

print("\nDuplicate Rows:")
print(sellers.duplicated().sum())

print("\nFirst Five Rows:")
display(sellers.head())

Shape: (3095, 4)

Columns:
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Data Types:


,Data Type
seller_id,object
seller_zip_code_prefix,int64
seller_city,object
seller_state,object



Missing Values:


,Missing
seller_id,0
seller_zip_code_prefix,0
seller_city,0
seller_state,0



Duplicate Rows:
0

First Five Rows:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [27]:
sellers.nunique()

seller_id                 3095
seller_zip_code_prefix    2246
seller_city                611
seller_state                23
dtype: int64

In [28]:
sellers["seller_id"].duplicated().sum()

np.int64(0)

In [29]:
print("Unique Cities:", sellers["seller_city"].nunique())
print("Unique States:", sellers["seller_state"].nunique())

Unique Cities: 611
Unique States: 23


In [30]:
sellers["seller_state"].value_counts()

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
PI       1
AC       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

In [31]:
print("Shape:", geolocation.shape)

print("\nColumns:")
print(geolocation.columns.tolist())

print("\nData Types:")
display(geolocation.dtypes.to_frame("Data Type"))

print("\nMissing Values:")
display(geolocation.isnull().sum().to_frame("Missing"))

print("\nDuplicate Rows:")
print(geolocation.duplicated().sum())

print("\nFirst Five Rows:")
display(geolocation.head())

Shape: (1000163, 5)

Columns:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Data Types:


,Data Type
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,object
geolocation_state,object



Missing Values:


,Missing
geolocation_zip_code_prefix,0
geolocation_lat,0
geolocation_lng,0
geolocation_city,0
geolocation_state,0



Duplicate Rows:
261831

First Five Rows:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [32]:
geolocation.nunique()

geolocation_zip_code_prefix     19015
geolocation_lat                717363
geolocation_lng                717615
geolocation_city                 8011
geolocation_state                  27
dtype: int64

In [33]:
geolocation[
    geolocation.duplicated()
].head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
72,1046,-23.545320,-46.644069,sao paulo,SP
79,1050,-23.549854,-46.643139,sao paulo,SP
80,1032,-23.540775,-46.635515,sao paulo,SP
82,1046,-23.546081,-46.644820,sao paulo,SP
86,1048,-23.547449,-46.640169,são paulo,SP


In [34]:
geolocation.nunique()

geolocation_zip_code_prefix     19015
geolocation_lat                717363
geolocation_lng                717615
geolocation_city                 8011
geolocation_state                  27
dtype: int64

In [35]:
geolocation.duplicated().sum()

np.int64(261831)

In [36]:
geolocation[
    geolocation.duplicated(keep=False)
].sort_values("geolocation_zip_code_prefix").head(20)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1004,1001,-23.549292,-46.633559,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
1435,1001,-23.549292,-46.633559,sao paulo,SP
912,1001,-23.550498,-46.634338,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP
99,1001,-23.549292,-46.633559,sao paulo,SP
851,1001,-23.549825,-46.633970,sao paulo,SP
639,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
1246,1001,-23.549292,-46.633559,sao paulo,SP


In [37]:
geolocation["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(981148)

In [38]:
geolocation["geolocation_zip_code_prefix"].value_counts().head(10)

geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
22631     832
30140     810
11740     788
38408     773
28970     743
Name: count, dtype: int64

In [39]:
geolocation[
    geolocation["geolocation_zip_code_prefix"] == 24220
]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
470805,24220,-22.905817,-43.106989,niteroi,RJ
470811,24220,-22.902306,-43.112545,niteroi,RJ
470812,24220,-22.904567,-43.110491,niteroi,RJ
470820,24220,-22.902575,-43.109192,niteroi,RJ
470821,24220,-22.907500,-43.106170,niteroi,RJ
...,...,...,...,...,...
474265,24220,-22.904023,-43.111683,niteroi,RJ
474266,24220,-22.905393,-43.100512,niterói,RJ
474269,24220,-22.906420,-43.104933,niteroi,RJ
474270,24220,-22.909701,-43.108452,niteroi,RJ


In [40]:
geolocation.duplicated()

0          False
1          False
2          False
3          False
4          False
           ...  
1000158    False
1000159     True
1000160     True
1000161    False
1000162     True
Length: 1000163, dtype: bool

In [41]:
geolocation = geolocation.drop_duplicates()

### Findings

- The geolocation dataset contains 1,000,163 records but only 19,015 unique ZIP code prefixes.
- Multiple latitude and longitude values exist for the same ZIP prefix, representing different geographic coordinates within the same postal region.
- 261,831 rows were exact duplicates across all columns and were safely removed.
- Repeated ZIP prefixes are expected because one ZIP prefix can map to multiple geographic coordinates.
- This table serves as a geographic lookup table that links ZIP code prefixes to cities, states, and coordinates.

In [42]:
print("Shape:", category_translation.shape)

print("\nColumns:")
print(category_translation.columns.tolist())

print("\nMissing Values:")
display(category_translation.isnull().sum().to_frame("Missing"))

print("\nDuplicate Rows:")
print(category_translation.duplicated().sum())

display(category_translation.head())

Shape: (71, 2)

Columns:
['product_category_name', 'product_category_name_english']

Missing Values:


,Missing
product_category_name,0
product_category_name_english,0



Duplicate Rows:
0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [43]:
category_translation.nunique()

product_category_name            71
product_category_name_english    71
dtype: int64

In [44]:
geolocation = geolocation.drop_duplicates()

In [45]:
print("Rows after removing duplicates:", geolocation.shape[0])
print("Remaining duplicate rows:", geolocation.duplicated().sum())

Rows after removing duplicates: 738332
Remaining duplicate rows: 0
